# Laboratorio 5 — Modelos de lenguaje

**Milton Beltrán** · Natural Language Processing

**Corpus:** *El ingenioso hidalgo don Quijote de la Mancha*, de Miguel de Cervantes
(Kaggle: [`manuelmaaf97/quijote`](https://www.kaggle.com/datasets/manuelmaaf97/quijote)).

A diferencia de los laboratorios #1 a #4, este no usa el corpus de noticias en español:
cambia de corpus y **de objetivo**. Ya no se clasifica un documento en una categoría, sino
que se predice la siguiente palabra de una secuencia. Eso invierte el pipeline de
normalización que se venía usando:

| Labs #1–#4 (clasificación) | Lab #5 (modelado de lenguaje) |
|---|---|
| Se eliminan las *stopwords* | Se conservan: son las palabras que un LM más predice |
| Stemming con `SnowballStemmer` | Ninguno: se predice la palabra de superficie |
| Bolsa de palabras, el orden no importa | El orden **es** el dato |
| Unidad de análisis: el documento | Unidad de análisis: la **oración**, con `<s>` y `</s>` |
| Partición estratificada por etiqueta | Partición aleatoria 80/10/10 (no hay etiquetas) |

Los modelos n-grama se implementan **desde cero** con `defaultdict`, según exige el
enunciado: no se usa `nltk.lm` ni `kenlm`. NLTK interviene únicamente para segmentar en
oraciones y tokenizar.

## 0. Preparación del entorno y carga del corpus

Descarga del corpus, lectura del texto crudo y recorte del aparato editorial de Project
Gutenberg. Al terminar esta sección hay una sola variable de interés, `texto`: el cuerpo
del libro como una cadena, todavía sin segmentar.

In [ ]:
# --- 0.1 Librerías ---

# NLTK 3.10 bloquea el import de `regex` cuando el .venv vive dentro del directorio de
# trabajo (falso positivo de su hook de seguridad). Esta variable lo desactiva y tiene
# que asignarse ANTES de `import nltk`.
import os

os.environ["NLTK_DISABLE_IMPORT_SECURITY"] = "1"

# --- Librerías estándar de Python ---
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

# --- Gráficos ---
import matplotlib.pyplot as plt

# --- NLP ---
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

# --- Descarga del corpus ---
import kagglehub

# Semilla global. La misma de los labs #3 y #4, por consistencia entre entregas.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)

# Tokens especiales de frontera de oración que pide la sección 1 del enunciado.
BOS, EOS = "<s>", "</s>"

print("Librerías cargadas · NLTK", nltk.__version__, "· kagglehub", kagglehub.__version__)

In [ ]:
# --- 0.2 Recursos de NLTK ---

# `punkt_tab` es el modelo del sentenizador Punkt, que es el que segmenta en oraciones.
# Punkt está entrenado por idioma: se le pasa language="spanish" al usarlo para que no
# confunda abreviaturas ni comillas del español con finales de oración.
for recurso in ["punkt", "punkt_tab"]:
    nltk.download(recurso, quiet=True)

print("Recursos de NLTK listos.")

In [ ]:
# --- 0.3 Descarga del corpus ---

# El dataset es público: `kagglehub` lo descarga sin credenciales y lo cachea en
# ~/.cache/kagglehub, así que solo baja la primera vez. Se optó por descargarlo en tiempo
# de ejecución en lugar de guardar una copia del .txt en el repositorio (decisión D-06 de
# la bitácora): la descarga es reproducible en cualquier máquina y evita arrastrar una
# quinta copia de un corpus al control de versiones.
ruta_dataset = Path(kagglehub.dataset_download("manuelmaaf97/quijote"))
RUTA_TXT = ruta_dataset / "don-quijote.txt"

print("Directorio del dataset:", ruta_dataset)
print("Archivo:", RUTA_TXT.name, f"({RUTA_TXT.stat().st_size / 1024**2:.1f} MB)")

In [ ]:
# --- 0.4 Lectura del texto crudo ---

# El archivo viene en UTF-8 *con BOM* y con saltos de línea CRLF (hallazgo H-02). Con
# encoding="utf-8" a secas, el carácter invisible \ufeff queda pegado al primer token del
# corpus; "utf-8-sig" lo consume. Los \r del CRLF los normaliza Python al leer en modo
# texto, así que no hay que tocarlos.
texto_crudo = RUTA_TXT.read_text(encoding="utf-8-sig")

print(f"Caracteres: {len(texto_crudo):,}")
print(f"Líneas:     {texto_crudo.count(chr(10)):,}")
print("¿Sobrevive el BOM?:", texto_crudo.startswith("\ufeff"))
print()
print("Primeros 200 caracteres:")
print(texto_crudo[:200])

In [ ]:
# --- 0.5 Recorte del aparato editorial de Project Gutenberg ---

# El archivo no es texto plano del Quijote: es el ebook #2000 de Project Gutenberg, con
# una cabecera y un pie de licencia EN INGLÉS (hallazgo H-01). Sin recortarlos, el modelo
# de lenguaje aprendería n-gramas como "Project Gutenberg-tm electronic works" dentro de
# un corpus que se supone es español del siglo XVII, contaminando el vocabulario, los
# conteos y la perplejidad.
#
# Se corta usando los marcadores y no números de línea fijos, para que el recorte no se
# rompa si el dataset cambia de versión (decisión D-07).
MARCA_INICIO = "*** START OF THIS PROJECT GUTENBERG EBOOK"
MARCA_FIN = "*** END OF THIS PROJECT GUTENBERG EBOOK"

ini = texto_crudo.index(MARCA_INICIO)
ini = texto_crudo.index("\n", ini) + 1  # saltar la línea del marcador
fin = texto_crudo.index(MARCA_FIN)

texto = texto_crudo[ini:fin].strip()

descartado = len(texto_crudo) - len(texto)
print(f"Antes del recorte:   {len(texto_crudo):>9,} caracteres")
print(f"Después del recorte: {len(texto):>9,} caracteres")
print(f"Descartado:          {descartado:>9,} caracteres ({descartado / len(texto_crudo):.1%})")

In [ ]:
# --- 0.6 Comprobación: ¿qué queda del aparato editorial? ---

# Recortar por los marcadores no basta: el crédito del voluntario que transcribió el libro
# está DENTRO de la región delimitada, igual que la línea de cierre (hallazgo H-05). Son
# pocas líneas, pero están en inglés y no son prosa de Cervantes.
restos = [linea.strip() for linea in texto.split("\n") if "Gutenberg" in linea]

print(f"Líneas con 'Gutenberg' que sobreviven al recorte: {len(restos)}\n")
for linea in restos:
    print("  ·", linea)

print("\nEl texto empieza ahora con:")
print(repr(texto[:130]))
print("\nY termina con:")
print(repr(texto[-130:]))

## 1. Preparación del corpus para modelado de secuencias

Pendiente. Pasos, en orden:

1. Descartar los restos editoriales detectados en 0.6 y decidir qué hacer con los **126
   encabezados de capítulo** y con los preliminares —TASA, privilegio real, versos
   dedicatorios— que no son prosa narrativa (decisión D-10, abierta en la bitácora).
2. **Reconstruir los párrafos.** El texto trae *wrap* duro a ~76 columnas: el salto de
   línea es tipográfico, no sintáctico, y una oración cruza varias líneas físicas
   (hallazgo H-03). Los párrafos están separados por líneas en blanco. Sentenizar sin
   reconstruirlos partiría casi todas las oraciones a la mitad.
3. Segmentar cada párrafo en oraciones con `sent_tokenize(..., language="spanish")`.
4. Tokenizar cada oración con `word_tokenize(..., language="spanish")`. **Sin** quitar
   *stopwords* y **sin** stemming.
5. Envolver cada oración entre `BOS` y `EOS`.
6. Partir las oraciones en 80 % entrenamiento / 10 % validación / 10 % prueba de forma
   aleatoria, con `RANDOM_STATE` fijo.
7. Reportar el tamaño del vocabulario de entrenamiento y la proporción de palabras de
   prueba que no aparecen en él (OOV).
8. Explicar la relación entre las palabras nunca vistas y la dispersión de datos.